# Scripting with the Implicit Solver

## Overview

Besides the input file, a simulation can be customized from a Python script. The following table provides an overview and the following sections are organized correspondingly.

| Task | Hook | Section |
|-|-|-|
| Prescribe or change the gap height | <code style="color:#c586c0">problem.topo.set_global_height</code>, <code style="color:#c586c0">problem.topo.set_mapped_height</code> | Custom Height |
| Impose boundary values by function | <code style="color:#c586c0">problem.set_bc_function</code> | Custom BC |
| Use your own material or wall-stress law | <code style="color:#c586c0">problem.set_custom_eos</code> | Custom Constitutive Laws |
| Stop and continue a simulation later | <code style="color:#c586c0">problem.save_state</code>, <code style="color:#c586c0">problem.load_state</code> | Checkpoints |
| Do things inside the Newton iteration | <code style="color:#c586c0">problem.add_callback(function)</code> | Solver subroutine |


## Gap height

The gap height is changed through `problem.topo`. Which function to use depends on where the height comes from:

| You have ... | Use | Array shape |
|-|-|-|
| a height profile, e.g. from a dataset | <code style="color:#c586c0">set_global_height(h)</code> | `topo.global_shape` |
| a function of $x, y, t$ | <code style="color:#c586c0">set_mapped_height(h)</code> | `topo.local_shape` |

Both functions also work in parallel runs: `set_global_height` distributes the array onto the processes (it only has to be valid on rank 0). For `set_mapped_height`, local helpers are provided: `topo.xx`, `topo.yy` (dimensional, [$\text{m}$]) or `topo.xx_norm`, `topo.yy_norm` (normalized to $[0, 1]$), which already have the required shape.

**Height profile from a dataset**

In [ ]:
from GaPFlow import Problem
import numpy as np

problem = Problem.from_yaml('config.yaml')

h_data = np.loadtxt('topo_roughness.txt')
assert h_data.shape == problem.topo.global_shape
problem.topo.set_global_height(h_data)

problem.run()

**Height changing in time**

Register a callback that updates the height after each time step. The change between two steps enters the equations as squeeze term, so activate <code style="color:#CE9178">implicit_solver.physics.squeeze</code>.

In [ ]:
problem = Problem.from_yaml('config.yaml')

h_base = np.ones(problem.topo.local_shape) * 1.5e-5

def oscillating_height():
    dh = 0.1 * problem.topo.xx_norm * np.sin(problem.simtime)
    problem.topo.set_mapped_height(h_base * (1 + dh))

problem.add_callback(oscillating_height)
problem.run()

**Helper attributes of** `problem.topo`

`global_shape` and `local_shape` are tuples. All other attributes are arrays of shape `local_shape`: they are local to the process and include the ghost layer.

| Attribute | Description |
|-|-|
| `global_shape` | Shape $(N_x, N_y)$ of the whole domain, without ghost cells. Required shape for `set_global_height`. |
| `local_shape` | Shape of the local array including ghost cells. Required shape for `set_mapped_height`. |
| `xx`, `yy` | Cell-centre coordinates in $[\text{m}]$, with the shape `local_shape`. |
| `xx_norm`, `yy_norm` | Normalized coordinates in $[0, 1]$, with the shape `local_shape`. |
| `h` | Current gap height in `local_shape`. Includes elastic deformation. |
| `dh_dx`, `dh_dy` | Slopes of the gap height in `local_shape`. |
| `deformation` | Elastic deformation, only non-zero with `properties.elastic` in `local_shape`. |

## Custom BC

Boundary values given as numbers in the input file can be replaced by a function at any boundary:

<code style="color:#c586c0">problem.set_bc_function(var, boundary, bc_function)</code>

| Argument | Values |
|-|-|
| `var` | `rho`, `jx`, `jy` |
| `boundary` | `W`, `E`, `S`, `N` |
| `bc_function` | function receiving a context `ctx` and returning an array of shape `ctx.required_shape` |

The context `ctx` provides:

| Attribute | Description |
|-|-|
| `problem` | The `Problem` instance (e.g. `ctx.problem.simtime`, `ctx.problem.q`). |
| `required_shape` | Shape of the array to return: the ghost layer at this boundary. |
| `slice_ghost` | Slice of the ghost layer in the fields of `problem.q`. |
| `slice_interior` | Slice of the adjacent interior layer in the fields of `problem.q`. |
| `x_norm`, `y_norm` | Normalized coordinates $[0,1]$ of the ghost layer. |

**Note:**
- The pressure is the unknown of the implicit solver. A function for `rho` therefore has to return **pressure values**, not densities.
- `jx` and `jy` functions return mass fluxes.
- In parallel runs a boundary can be split between processes. Always build the result from `required_shape`, `x_norm`, `y_norm` or from `problem.q[...][slice_...]`, never from global indices.

A uniform, time-dependent pressure at the west boundary:

In [ ]:
def oscillating_pressure(ctx):
    p_bc = 1e5 * (1 + 0.1 * np.sin(ctx.problem.simtime))
    return np.full(ctx.required_shape, p_bc)

problem.set_bc_function('rho', 'W', oscillating_pressure)

A Couette profile $j_x = \rho\, u(y)$ at the west boundary, with $u$ increasing from 0 to $U_\mathrm{max}$ along $y$:

In [ ]:
U_max = 10.  # m/s

def couette_profile(ctx):
    u = U_max * ctx.y_norm
    rho = ctx.problem.q[0][ctx.slice_interior]
    return rho * u

problem.set_bc_function('jx', 'W', couette_profile)

## Custom Constitutive Laws

Replace built-in material and wall-stress models by user functions, set on the `problem` instance, e.g.

- <code style="color:#c586c0">problem.set_custom_eos(function)</code> - see `Customized EoS` in [this notebook](B08_fem_examples.ipynb)
- <code style="color:#c586c0">problem.set_custom_delta_tau_xz(function)</code> - TODO

## Checkpoints

A checkpoint stores everything needed to continue a simulation seamlessly: all solution fields (including ghost cells), the current gap height and, if enabled, elastic deformation and force-balance state, as well as time, time step and convergence history. It is independent of the regular output files (`sol.nc` etc.).

| Function | Description |
|-|-|
| <code style="color:#c586c0">problem.save_state(path)</code> | Write the checkpoint to a single `.npz` file. |
| <code style="color:#c586c0">problem.load_state(path)</code> | Restore it. Call on a freshly constructed `Problem` **before** `run()`. |

Rules for a restart:
- Use the same input file (grid, geometry, properties, `implicit_solver`) as for the saved run.
- The number of MPI processes may differ.
- Only available for the implicit solver.

In [ ]:
# first run
problem = Problem.from_yaml('config.yaml')
problem.run()
problem.save_state('checkpoint.npz')

# later: continue
problem = Problem.from_yaml('config.yaml')
problem.load_state('checkpoint.npz')
problem.run()

`problem.step` and `problem.simtime` are restored, so <code style="color:#CE9178">numerics.max_it</code> counts the total number of steps, including those of the first run.

## Solver subroutine

Functions registered with <code style="color:#c586c0">problem.add_callback(function)</code> are called at $t=0$ and after each time step. They take no arguments and access all data through the `problem` instance (closure). Use them for monitoring or any action that should happen every step.

The example records the total mass and the maximum speed (both are summed/maximized over all processes in parallel runs) and writes a checkpoint every 100 steps:

In [ ]:
history = {'step': [], 'time': [], 'mass': [], 'v_max': []}

def track():
    history['step'].append(problem.step)
    history['time'].append(problem.simtime)
    history['mass'].append(problem.mass)
    history['v_max'].append(problem.v_max)

    if problem.step > 0 and problem.step % 100 == 0:
        problem.save_state(f'checkpoint_{problem.step:06d}.npz')

problem.add_callback(track)
problem.run()

## Parallel runs

When following the rules above, the customizations are consistent in parallel mode.

Launch a script with MPI:

```bash
mpirun -n 4 python my_script.py
```

The domain is split between the processes, each of which only sees its local subdomain. The decomposition is available as `problem.decomp`:

| Attribute | Description |
|-|-|
| `rank`, `size` | MPI rank of this process, total number of processes |
| `is_at_xW`, `is_at_xE`, `is_at_yS`, `is_at_yN` | Whether this process owns the respective domain boundary |

Fields can be collected on rank 0 for post-processing with <code style="color:#c586c0">problem.decomp.gather_global(field)</code>. It is a collective call: all processes must call it, and only rank 0 receives the global array.

In [ ]:
rho_global = problem.decomp.gather_global(problem.q[0])

if problem.decomp.rank == 0:
    ...  # plot or analyze rho_global